## データ読み込み

In [1]:
import json

with open("data/article_likes2.json", encoding="utf-8") as f:
    likes_data = json.load(f)

In [2]:
def summarize_likes(article_likes):
    # 取得済みのいいね情報から件数を再計算（取得漏れ確認のため）
    like_counts = [
        len(article["likes"])
        for article in article_likes.values()
    ]

    users = set()

    for article in article_likes.values():
        for like in article["likes"]:
            users.add(like["user"]["id"])

    print("記事数:", len(article_likes))
    print("最大いいね数:", max(like_counts))
    print("平均いいね数:", sum(like_counts) / len(like_counts))
    print("ユーザ数:", len(users))

summarize_likes(likes_data)

記事数: 886
最大いいね数: 577
平均いいね数: 21.959367945823928
ユーザ数: 7345


In [3]:
def filter_articles_by_likes(
    likes_data: dict,
    min_likes: int,
) -> dict:
    """いいね数が指定値以上の記事を除外する。"""

    return {
        article_id: article
        for article_id, article in likes_data.items()
        if len(article["likes"]) < min_likes
    }

# 超人気記事の影響を無くすため、いいね100以上の記事を除外する
likes_data = filter_articles_by_likes(likes_data, 100)

## ネットワーク作成

In [4]:
import networkx as nx


def create_like_network(article_likes: dict) -> nx.DiGraph:
    """
    いいね情報からユーザ→記事の有向ネットワークを作成する。

    ノード:
        記事:
            node id: article:{記事ID}
            attributes:
                type: article
                id: 記事ID
                title: 記事名

        ユーザ:
            node id: user:{ユーザID}
            attributes:
                type: user
                id: ユーザID
                name: ユーザ名
                organization: Organization名

    エッジ:
        user → article
        ユーザが記事にいいねしたことを表す
    """

    G = nx.DiGraph()

    for article_id, article in article_likes.items():

        article_node = f"article:{article_id}"

        # 記事ノード追加
        G.add_node(
            article_node,
            type="article",
            id=article_id,
            title=article["title"],
            organization=article["author_organization"]
        )

        # いいねしたユーザ
        for like in article["likes"]:
            user = like["user"]

            user_id = user["id"]
            user_node = f"user:{user_id}"

            # ユーザノード追加
            if user_node not in G:
                G.add_node(
                    user_node,
                    type="user",
                    id=user_id,
                    name=user["name"],
                    organization=user.get("organization", ""),
                )

            # ユーザ → 記事
            G.add_edge(
                user_node,
                article_node,
                type="like",
            )

    return G

In [5]:
G = create_like_network(likes_data)

print(G.nodes["article:79c4e7c3df9b6b61bbb4"])

{'type': 'article', 'id': '79c4e7c3df9b6b61bbb4', 'title': '「基本情報技術者試験をRPGで攻略する無料Webアプリを個人開発した話」', 'organization': None}


In [6]:
def get_graph_statistics(G: nx.DiGraph) -> dict:
    """グラフの基本統計量を取得する。"""

    article_nodes = [
        node for node, data in G.nodes(data=True)
        if data["type"] == "article"
    ]

    user_nodes = [
        node for node, data in G.nodes(data=True)
        if data["type"] == "user"
    ]

    return {
        # ノード
        "node_count": G.number_of_nodes(),
        "article_count": len(article_nodes),
        "user_count": len(user_nodes),

        # エッジ
        "edge_count": G.number_of_edges(),

        # 密度
        "density": nx.density(G),

        # 次数
        "average_degree": (
            sum(dict(G.degree()).values()) / G.number_of_nodes()
            if G.number_of_nodes() > 0
            else 0
        ),

        # ユーザー → 記事のエッジ
        "like_count": sum(
            1
            for _, _, data in G.edges(data=True)
            if data.get("type") == "like"
        ),
    }

stats = get_graph_statistics(G)

print(stats)

{'node_count': 5083, 'article_count': 843, 'user_count': 4240, 'edge_count': 10942, 'density': 0.0004235863338397633, 'average_degree': 4.305331497147354, 'like_count': 10942}


## 中心性を求める

In [8]:
import networkx as nx
import pandas as pd


def calculate_centralities(G: nx.DiGraph):
    """
    ネットワークの中心性を計算する。

    無向グラフにして求める。
    """

    UG = G.to_undirected()

    betweenness = nx.betweenness_centrality(UG)
    closeness = nx.closeness_centrality(UG)
    eigenvector = nx.eigenvector_centrality(
        UG,
        max_iter=1000,
        tol=1e-06,
    )

    return {
        "betweenness": betweenness,
        "closeness": closeness,
        "eigenvector": eigenvector,
    }


def create_article_df(
    G: nx.DiGraph,
    centralities: dict,
) -> pd.DataFrame:
    """
    記事ノードから記事DataFrameを作成する。

    columns:
        id
        title
        organization
        likes
        betweenness
        closeness
        eigenvector
    """

    rows = []

    for node, data in G.nodes(data=True):

        if data["type"] != "article":
            continue

        # 記事に向かうエッジ数 = いいね数
        likes = G.in_degree(node)

        rows.append(
            {
                "id": data["id"],
                "title": data["title"],
                "organization": data.get("organization", ""),
                "likes": likes,
                "betweenness": centralities["betweenness"].get(node, 0),
                "closeness": centralities["closeness"].get(node, 0),
                "eigenvector": centralities["eigenvector"].get(node, 0),
            }
        )

    return pd.DataFrame(rows)


def create_user_df(
    G: nx.DiGraph,
    centralities: dict,
    article_likes: dict,
) -> pd.DataFrame:
    """
    ユーザーノードからユーザーDataFrameを作成する。

    columns:
        id
        name
        organization
        likes
        items
        betweenness
        closeness
        eigenvector
    """

    # ユーザーごとの投稿数を取得
    # article_likes側にユーザー情報があるので、
    # user_id -> items_count の辞書を作る
    user_items_count = {}

    for article in article_likes.values():
        for like in article["likes"]:
            user = like["user"]
            user_id = user["id"]

            user_items_count[user_id] = user.get(
                "items_count",
                0,
            )

    rows = []

    for node, data in G.nodes(data=True):

        if data["type"] != "user":
            continue

        # ユーザーから記事へのエッジ数 = いいねした記事数
        likes = G.out_degree(node)

        user_id = data["id"]

        rows.append(
            {
                "id": user_id,
                "name": data["name"],
                "organization": data.get("organization", ""),
                "likes": likes,
                "items": user_items_count.get(user_id, 0),
                "betweenness": centralities["betweenness"].get(node, 0),
                "closeness": centralities["closeness"].get(node, 0),
                "eigenvector": centralities["eigenvector"].get(node, 0),
            }
        )

    return pd.DataFrame(rows)


def create_analysis_dataframes(
    article_likes: dict,
    G: nx.DiGraph,
):
    """
    Qiitaいいねネットワークから記事・ユーザーの
    DataFrameを作成する。

    Returns:
        article_df
        user_df
    """

    # 中心性をまとめて計算
    centralities = calculate_centralities(G)

    # 記事DataFrame
    article_df = create_article_df(
        G,
        centralities,
    )

    # ユーザーDataFrame
    user_df = create_user_df(
        G,
        centralities,
        article_likes,
    )

    return article_df, user_df

In [9]:
article_df, user_df = create_analysis_dataframes(
    likes_data,
    G,
)

### 中心性を分析する

### 全体

In [11]:
def get_article_rankings(
    article_df: pd.DataFrame,
    top_n: int = 5,
) -> dict[str, pd.DataFrame]:
    """記事のいいね数・各中心性ランキングを取得する。"""

    ranking_columns = [
        "likes",
        "betweenness",
        "closeness",
        "eigenvector",
    ]

    display_columns = [
        "id",
        "title",
        "organization",
        "likes",
    ]

    rankings = {}

    for column in ranking_columns:
        columns = display_columns.copy()

        if column not in columns:
            columns.append(column)

        rankings[column] = (
            article_df[columns]
            .sort_values(column, ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    return rankings
    
rankings = get_article_rankings(article_df, 10)

print("=== いいね数 ===")
print(rankings["likes"])

print("=== 媒介中心性 ===")
print(rankings["betweenness"])

print("=== 近接中心性 ===")
print(rankings["closeness"])

print("=== 固有ベクトル中心性 ===")
print(rankings["eigenvector"])

=== いいね数 ===
                     id                                              title  \
0  cd2c603abf8b48fc23a8                      AIが書くpandasコード、だいたい地雷が混じっている話   
1  03b9b817bf2015321159                  プログラミング完全未経験から始める！競技プログラミング入門ガイド    
2  c63073fef7fb87d9f472    PRを出す前にコミット履歴を整えろと先輩に叩き込まれた話 〜git rebase -i 入門〜   
3  fa1f6b0db92e497796ac                      会社が変わっても消えないスキルとスタートアップという選択肢   
4  01defe8188de2123b25c                        AI時代、“ITを知らない”が普通に危険になってきた話   
5  996735e9b8ff4b3b1255                            要件定義とは何か？開発メンバー初心者向けガイド   
6  9acccd181ca2f1a75f3c  Copilot StudioをClaude Code化したら、Copilot Studio自...   
7  34010e35cd35c797d258                    【35歳未経験でも理解できた】Ruby on Rails 前編   
8  6fbf02174f308e31f284  ClaudeCodeとCodexにコーディングを全て任せて商用レベルのUnityゲーム開発を...   
9  f1855ff508f4268df5b5                   私の最強のMac開発環境 2026: Nixとmiseで育てる🐱   

  organization  likes  
0    FPT_Japan     90  
1          NaN     89  
2     beex-inc     88  
3         prum     87  
4       

### いいね

In [12]:
(article_df
.sort_values("likes", ascending=False)
.reset_index(drop=True)
).head(20)

,id,title,organization,likes,betweenness,closeness,eigenvector
0,cd2c603abf8b48fc23a8,AIが書くpandasコード、だいたい地雷が混じっている話,FPT_Japan,90,0.038372,0.292240,0.006295
1,03b9b817bf2015321159,プログラミング完全未経験から始める！競技プログラミング入門ガイド,NaN,89,0.028061,0.268560,0.005895
2,c63073fef7fb87d9f472,PRを出す前にコミット履歴を整えろと先輩に叩き込まれた話 〜git rebase -i 入門〜,beex-inc,88,0.029348,0.282813,0.007733
3,fa1f6b0db92e497796ac,会社が変わっても消えないスキルとスタートアップという選択肢,prum,87,0.018593,0.271903,0.008680
4,01defe8188de2123b25c,AI時代、“ITを知らない”が普通に危険になってきた話,prum,86,0.019713,0.270850,0.007160
5,996735e9b8ff4b3b1255,要件定義とは何か？開発メンバー初心者向けガイド,xincere-inc,85,0.023608,0.273484,0.007527
6,9acccd181ca2f1a75f3c,Copilot StudioをClaude Code化したら、Copilot Studio自...,ibm,81,0.028282,0.269983,0.003892
7,34010e35cd35c797d258,【35歳未経験でも理解できた】Ruby on Rails 前編,prum,78,0.011452,0.252507,0.005435
8,6fbf02174f308e31f284,ClaudeCodeとCodexにコーディングを全て任せて商用レベルのUnityゲーム開発を...,NaN,73,0.028395,0.271632,0.004211
9,f1855ff508f4268df5b5,私の最強のMac開発環境 2026: Nixとmiseで育てる🐱,nekonata,72,0.025855,0.280092,0.005063


### 媒介

In [13]:
(article_df
.sort_values("betweenness", ascending=False)
.reset_index(drop=True)
).head(20)

,id,title,organization,likes,betweenness,closeness,eigenvector
0,cd2c603abf8b48fc23a8,AIが書くpandasコード、だいたい地雷が混じっている話,FPT_Japan,90,0.038372,0.292240,0.006295
1,1b3853a3314ab66eb2a3,TransformerのSelf AttentionのQKVを直感的に解説する,NaN,72,0.030114,0.287230,0.005042
2,c63073fef7fb87d9f472,PRを出す前にコミット履歴を整えろと先輩に叩き込まれた話 〜git rebase -i 入門〜,beex-inc,88,0.029348,0.282813,0.007733
3,026452114ad63efa9aa6,果たしてCopilotで営業成績は上がるのか,NaN,55,0.028658,0.284557,0.004818
4,6fbf02174f308e31f284,ClaudeCodeとCodexにコーディングを全て任せて商用レベルのUnityゲーム開発を...,NaN,73,0.028395,0.271632,0.004211
5,9acccd181ca2f1a75f3c,Copilot StudioをClaude Code化したら、Copilot Studio自...,ibm,81,0.028282,0.269983,0.003892
6,03b9b817bf2015321159,プログラミング完全未経験から始める！競技プログラミング入門ガイド,NaN,89,0.028061,0.268560,0.005895
7,f1855ff508f4268df5b5,私の最強のMac開発環境 2026: Nixとmiseで育てる🐱,nekonata,72,0.025855,0.280092,0.005063
8,996735e9b8ff4b3b1255,要件定義とは何か？開発メンバー初心者向けガイド,xincere-inc,85,0.023608,0.273484,0.007527
9,39e8e3bbc327526ac20f,単一HTMLで作ったサイトを自己解凍形式にする試み,NaN,52,0.023197,0.290195,0.005275


### 近接

In [14]:
(article_df
.sort_values("closeness", ascending=False)
.reset_index(drop=True)
).head(40)

,id,title,organization,likes,betweenness,closeness,eigenvector
0,1b5b5016b5a74fe27fe5,OpenAIがCodexを無料開放——これ、何が目的なんだろう,tomosia,48,0.020836,0.292345,0.006950
1,cd2c603abf8b48fc23a8,AIが書くpandasコード、だいたい地雷が混じっている話,FPT_Japan,90,0.038372,0.292240,0.006295
2,39e8e3bbc327526ac20f,単一HTMLで作ったサイトを自己解凍形式にする試み,NaN,52,0.023197,0.290195,0.005275
3,1b3853a3314ab66eb2a3,TransformerのSelf AttentionのQKVを直感的に解説する,NaN,72,0.030114,0.287230,0.005042
4,026452114ad63efa9aa6,果たしてCopilotで営業成績は上がるのか,NaN,55,0.028658,0.284557,0.004818
5,c63073fef7fb87d9f472,PRを出す前にコミット履歴を整えろと先輩に叩き込まれた話 〜git rebase -i 入門〜,beex-inc,88,0.029348,0.282813,0.007733
6,d7a26696d724f6e83830,社内カリキュラムが微妙だったので、「その前提知らないんだけど…」を解決する学習サイトを作った,NaN,33,0.014010,0.281187,0.005887
7,a86a42a675e27853299b,Claude Code入りのDockerイメージをDevContainerで動かす,NaN,37,0.015537,0.280542,0.004398
8,20bc6f13035ac3ebf468,Codex開発で収益化するまで#4,NaN,13,0.005162,0.280157,0.004187
9,f1855ff508f4268df5b5,私の最強のMac開発環境 2026: Nixとmiseで育てる🐱,nekonata,72,0.025855,0.280092,0.005063


### 固有ベクトル

In [15]:
(article_df
.sort_values("eigenvector", ascending=False)
.reset_index(drop=True)
).head(40)

,id,title,organization,likes,betweenness,closeness,eigenvector
0,75b8ff55095604bd48b5,PythonとJavaのコードの違い,any-plus,56,5.984536e-03,0.261245,0.148875
1,f02c34ae6dc994cd3084,【新入社員研修】Linuxの基本操作,any-plus,54,5.955867e-03,0.263468,0.148605
2,d39b79afdbfadf5a2267,Pythonの制御構造について【基礎】,any-plus,47,2.061160e-03,0.254160,0.146757
3,a3e11fa3d865748a5610,ビジネスマナーは“形”だけじゃない——新卒研修で学んだことと考えたこと,any-plus,49,2.984651e-03,0.255380,0.146540
4,c5d039e4efec7f0fb3f7,『入社１年目の教科書』 意識したい３つの原則,any-plus,46,2.443174e-03,0.256748,0.146173
5,46fd3c7066b0f3c1b13e,報連相の重要性,any-plus,48,3.068058e-03,0.258594,0.145842
6,6303ef4ef36779011972,文系未経験エンジニアが研修で学んだLinuxコマンド備忘録,any-plus,61,7.836424e-03,0.261692,0.145565
7,e7939e9f2a8b3e8177f8,[初心者]Linuxと仲良くなれるコマンド3選,any-plus,53,4.095105e-03,0.253422,0.145354
8,0675640c728125a628c4,Linuxのテキストエディタ viとnano,any-plus,45,1.398766e-03,0.247775,0.144878
9,f493dac42a3efaf4ec42,Cisco IOSモードの超ザックリとした覚え方,any-plus,46,4.703957e-04,0.194544,0.144166
